<a href="https://colab.research.google.com/github/Jules-Vatel/SSI_SPRING/blob/main/SSI_SPRING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
url = "https://raw.githubusercontent.com/Jules-Vatel/SSI_SPRING/refs/heads/main/Offer-Westort_April%2028%2C%202026_08.08.csv"
df = pd.read_csv(url)


In [33]:
#function for seven point party Id.
def seven_pt_pid(row):
  party = row['Q-PartyAffiliation']
  strength = row['Q-PolarAffiliation']
  lean = row['Q-MiddleAffiliation']
  if (party=='Democrat'):
      return 1 if strength == 'Strong' else 2
  elif(party=='Republican'):
      return 7 if strength == 'Strong' else 6
  elif(party in ['Independent','No Preference','Other Party (Please Specify)']):
      if (lean=='Closer to Democratic Party'):
          return 3
      elif (lean=='Closer to Republican Party'):
          return 5
      elif(lean=='Neither'):
          return 4
  return np.nan


In [34]:
# function that gets party for leaners too
def get_party(row):
    pa = row['Q-PartyAffiliation']
    if(pa=='Democrat'):
        return 'Democrat'
    elif(pa=='Republican'):
        return 'Republican'
    elif(pa in ['Independent','No Preference','Other Party (Please Specify)']):
        ma = row.get('Q-MiddleAffiliation', np.nan)
        if(ma=='Closer to Democratic Party'):
            return 'Democrat'
        elif(ma=='Closer to Republican Party'):
            return 'Republican'
    return np.nan


In [35]:
#tuple used to map the questions evaluating how comfortable respondents were to certain policies,
#relates to the social distance,close freinds distance, and inlaw distance questions,
#used to quantize each of the values

comfort_map = {
    'Very uncomfortable': 1, 'Uncomfortable': 2, 'Somewhat uncomfortable': 3,
    'Neither comfortable nor uncomfortable': 4, 'Somewhat comfortable': 5,
    'Comfortable': 6, 'Very comfortable': 7, 'Very Comfortable': 7
}

#same thing as comfort map but relates to the relection question so it can be mapped to varaibles
reelect_map = {
    'Very unlikely': 1, 'Unlikely': 2, 'Somewhat unlikely': 3,
    'Neither likely nor unlikely': 4, 'Somewhat likely': 5,
    'Likely': 6, 'Very likely': 7
}


#we can change these if we feel necessary

In [58]:

#made a class to navigate the data better
#have access to each individual model,a 2d array containing all the models in a 2d array
# the class also has funciton that print the results from the regression direclty, I will add plotting functions and some other functions
#parameters to the constructor are a comfort_map for the social distance ,close freinds distance, inlaw distace questions and
# the relect map for the Q-ReElelection collumn
#affective polarization measured by first applying the comfort map to the collumns in question, these are added to a filler column with a _n added to the name
#example row#1 has temporary row row#1_n. Then the mean score of the three is taken and subtracted by 8 to get the Post treatment outcome varialbe.

def run_ols(formula,data):
  return sm.OLS.from_formula(formula,data).fit(cov_type='HC1')

class DATA_ANALYSIS:
  def __init__(self,df=df,cmap=comfort_map,rmap=reelect_map):
    EQ_AP1 = ""
    df = df[-df['Treat_Party'].str.contains('ImportId',na=False)].copy()
    df['Feeling_ThermR'] = pd.to_numeric(df['Q-FeelingThermR_1'], errors='coerce')
    df['Feeling_ThermD'] = pd.to_numeric(df['Q-FeelingThermD_1'], errors='coerce')
    df['PA']=df.apply(seven_pt_pid,axis=1)
    df['Party'] = df.apply(get_party, axis=1)
    df['ThermInP']  = np.where(df['Party'] == 'Democrat', df['Feeling_ThermD'], df['Feeling_ThermR'])
    df['ThermOutP'] = np.where(df['Party'] == 'Democrat', df['Feeling_ThermR'], df['Feeling_ThermD'])
    # get pre-treatment affective polarization
    df['APpre'] = (df['ThermInP'] - df['ThermOutP']).abs()
    for col in ['Q-InLawDist', 'Q-CloseFriendDist', 'Q-Social Distance 3']:
      df[col + '_n'] = df[col].map(cmap)
    # get affective polarization after as additive/mean value
    df['APpost'] = df[['Q-InLawDist_n', 'Q-CloseFriendDist_n','Q-Social Distance 3_n']].mean(axis=1)
    df['TB'] = df['Q-ReElection'].map(rmap)
    df['Treat'] = (df['Treat_Party'] != 'Control').astype(int)
    df['Pin']   = (df['Treat_Party'] == 'InParty').astype(int)
    df['FE']    = (df['Treat_Frame'] == 'Electoral').astype(int)
    df['FD']    = (df['Treat_Frame'] == 'Democracy').astype(int)
    analysis_vars = ['APpost','TB','APpre','PA']
    full = df.dropna(subset=analysis_vars).copy()
    treated = full[full['Treat']==1].copy()
    AP_1 = run_ols('APpost ~ Treat + APpre + PA', full)
    TB_1 = run_ols('TB ~ Treat + APpre + PA', full)
    AP_2 = run_ols('APpost ~ Pin + APpre + PA', treated)
    TB_2 = run_ols('TB ~ Pin + APpre + PA', treated)
    AP_3 = run_ols('APpost ~ FE + FD + Pin + APpre + PA', treated)
    TB_3 = run_ols('TB ~ FE + FD + Pin + APpre + PA', treated)
    AP_4 = run_ols('APpost ~ FE + FD + Pin + FE:Pin + FD:Pin + APpre + PA', treated)
    TB_4 = run_ols('TB ~ FE + FD + Pin + FE:Pin + FD:Pin + APpre + PA', treated)
    self.AP_1,self.AP_2,self.AP_3,self.AP_4 = AP_1,AP_2,AP_3,AP_4
    self.TB_1,self.TB_2,self.TB_3,self.TB_4 = TB_1,TB_2,TB_3,TB_4
    ftest_AP3 = AP_3.f_test("FE = FD")
    ftest_TB3 = TB_3.f_test("FE = FD")
    ftest_AP4 = AP_4.f_test("FE:Pin = 0, FD:Pin = 0")
    ftest_TB4 = TB_4.f_test("FE:Pin = 0, FD:Pin = 0")
    self.models=[[AP_1,TB_1],[AP_2,TB_2],[AP_3,TB_3],[AP_4,TB_4]]
    self.ftests=[[ftest_AP3,ftest_TB3],[ftest_AP4,ftest_TB4]]
    self.samples_full = len(full)
    self.samples_treated = len(treated)

  def print_summary_all(self):
    labels=["Affective Polarization","Tolarance for Backsliding"]
    models = self.models
    for i in range(len(models)):
      for j in range(len(models[i])):
        print(f"Model#{i+1} Results ({labels[j]})")
        print(models[i][j].summary(),"\n")

  def print_summary_single(self,model_num,id):
    models = self.models
    if(model_num>len(models)):
      raise ValueError(f"Not A Valid Model Number: There are {len(models)} models")
    n = None
    mn = model_num-1
    labels=["Affective Polarization","Tolarance for Backsliding"]
    if(id.upper()=="AP"):
      n = 0
    else:
      n = 1
    print(f"Model#{model_num} Results ({labels[n]})")
    print(models[mn][n].summary(),"\n")

  def print_summary_model(self,model_num):
    models = self.models
    labels=["Affective Polarization","Tolarance for Backsliding"]
    if(model_num>len(models[0])):
      raise ValueError(f"Not A Valid Model Number: There are {models[0].len()} models")
    mn = model_num-1
    for i in range(len(models[mn])):
      print(f"Model#{model_num} Results ({labels[i]})")
      print(models[mn][i].summary(),"\n")

    def graph_delta_ap(self):
      models = self.models()
      fig,ax = plt.subplots()
      x = ["Model #1","Model #2","Model #3","Model #4"]
      y = []














In [57]:

results = DATA_ANALYSIS()
#results.print_summary_single(1,"AP")
results.print_summary_all()
#results.print_summary_model(1)




Model#1 Results (Affective Polarization)
                            OLS Regression Results                            
Dep. Variable:                 APpost   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.031
Method:                 Least Squares   F-statistic:                   0.02549
Date:                Thu, 30 Apr 2026   Prob (F-statistic):              0.994
Time:                        15:33:23   Log-Likelihood:                -128.30
No. Observations:                  97   AIC:                             264.6
Df Residuals:                      93   BIC:                             274.9
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  